In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from util import filter_data, seed_everything, report_split_stats
from util import mask_crop as mask_crop_fn
from validate import val_model, ValLoaderWrapper
from loader import QSM_c1_Dataset as QSM_RAM_Dataset
from networks import QSMDecoder, ResNetWrapper

# ============================================================
# EXPERIMENT CONFIG
# ============================================================
EXP_NAME = "cond_pretext_no_memorize" 
device = 'cuda:0'
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_preprocessed_cache.pt'
LOAD_FROM_CACHE = True 
seed = 0
seed_everything(0)

# ============================================================
# RUNTIME & CV
# ============================================================
nii_path = '/data2/ali/dbs/qsm/'
seg_path = '/data2/ali/dbs/seg_ps/'
file_dir = '/data2/ali/dbs/dbs_03292024.csv'

cv_features = {'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', ' pre op levadopa equivalent dose (mg)', ' Test medication status', ' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)'}
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}

motor_df = filter_data(file_dir, all_needed_cols, True)

motor_df[' OFF (pre-dbs updrs)'] = pd.to_numeric(motor_df[' OFF (pre-dbs updrs)'], errors='coerce')
motor_df[' ON (pre-dbs updrs)'] = pd.to_numeric(motor_df[' ON (pre-dbs updrs)'], errors='coerce')
motor_df[' OFF meds ON stim 6mo'] = pd.to_numeric(motor_df[' OFF meds ON stim 6mo'], errors='coerce')
motor_df = motor_df.dropna(subset=[' OFF (pre-dbs updrs)', ' OFF meds ON stim 6mo'])
raw_motor_df = motor_df.copy()
improvement_ratios = (motor_df[' OFF (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' OFF (pre-dbs updrs)']
label_map = {int(row['CORNELL ID']): (1 if ratio >= 0.30 else 0) for (_, row), ratio in zip(motor_df.iterrows(), improvement_ratios)}
# Clinical data normalization
cols_to_norm = ['Age', 'Disease Duration (year)', ' OFF (pre-dbs updrs)', ' pre op levadopa equivalent dose (mg)']

for col in cols_to_norm:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')
    col_mean = motor_df[col].mean()
    motor_df[col] = motor_df[col].fillna(col_mean)
    col_std = motor_df[col].std()
    motor_df[col] = (motor_df[col] - col_mean) / (col_std + 1e-8)

print(">>> Clinical features normalized successfully.")
clinical_dict = {str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) 
                 for _, row in motor_df.iterrows()}
full_dataset = QSM_RAM_Dataset(nii_path, seg_path, mask_crop_fn, clinical_dict, label_map, limit=LIMIT_SUBS, cache_path=CACHE_PATH, load_cache=LOAD_FROM_CACHE)
actual_clin_dim = next(iter(clinical_dict.values())).shape[0]
full_dataset.clin_dim = actual_clin_dim

all_cached_ids = set(full_dataset.volumes.keys())
labeled_subs = np.array(list(all_cached_ids & set(label_map.keys())))
unlabeled_ids = np.array(list(all_cached_ids - set(label_map.keys())))
sub_labels = np.array([label_map[sid] for sid in labeled_subs])

qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))
])

print(f">>> Pretraining still using {len(all_cached_ids)} total volumes.")
all_split_best_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for split, (t_p_idx, v_p_idx) in enumerate(skf.split(labeled_subs, sub_labels)):
    train_subs, val_subs = labeled_subs[t_p_idx], labeled_subs[v_p_idx]
    report_split_stats(train_subs, val_subs, raw_motor_df)
    pt_subs = np.concatenate([unlabeled_ids, train_subs])
    
    t_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in train_subs]
    v_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in val_subs]
    pt_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in pt_subs]

    pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)
    train_labels = [label_map[full_dataset.samples[i]['sub_id']] for i in t_idx]
    class_weights = 1. / torch.tensor(np.bincount(train_labels), dtype=torch.float)
    sampler = WeightedRandomSampler([class_weights[l] for l in train_labels], 2*len(t_idx))
    t_loader = DataLoader(Subset(full_dataset, t_idx), batch_size=48, sampler=sampler)
    v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

    print(f"\n>>> Split {split} | PT Subs: {len(pt_subs)} | Val Subs: {len(val_subs)}")
    # --- PRE-AUGMENTATION SUMMARY ---
    total_subs = len(labeled_subs)
    non_responders = np.sum(sub_labels == 0)
    responders = np.sum(sub_labels == 1)

    print("\n" + "!"*30)
    print("PRE-AUGMENTATION COHORT SUMMARY")
    print(f"Total Unique Subjects: {total_subs}")
    print(f"Total Non-Responders (0): {non_responders}")
    print(f"Total Responders (1):     {responders}")
    print(f"Baseline Response Rate:  {(responders/total_subs):.1%}")
    print("!"*30 + "\n")
    # --- A. CONDITIONAL PRETRAINING ---
    base_resnet = models.resnet18(weights='IMAGENET1K_V1')
    base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    model = ResNetWrapper(base_resnet, clinical_dim=actual_clin_dim).to(device)
    decoder = QSMDecoder(feat_dim=512).to(device)
    
    # Optimizer: Only update Backbone + Decoder. Skip 'fusion' to avoid identity memorization.
    optimizer_pt = torch.optim.Adam(list(model.base_model.parameters()) + list(decoder.parameters()), lr=1e-4)
    criterion_pt = nn.MSELoss()

    full_dataset.train_mode, full_dataset.transform = True, qsm_aug
    for pt_epoch in range(5):
        model.base_model.train(); decoder.train()
        for imgs, clin, _, _ in pt_loader:
            imgs, clin = imgs.to(device), clin.to(device)
            optimizer_pt.zero_grad()
            
            # Use visual features for reconstruction
            _, feats = model(imgs, clin)
            recon = decoder(feats)
            
            loss = criterion_pt(recon, imgs)
            loss.backward()
            optimizer_pt.step()

    # --- B. FINE-TUNING ---
    print(">>> Fine-tuning Classifier Head (Backbone Frozen)")
    for param in model.base_model.parameters(): param.requires_grad = False
    
    optimizer = torch.optim.Adam(model.fusion.parameters(), lr=5e-5, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss().to(device)

    best_f1, patience, best_metrics_this_split = 0, 0, None
    os.makedirs(f"weights/{EXP_NAME}", exist_ok=True)

    for epoch in range(30):
        model.train(); full_dataset.train_mode = True
        for imgs, clin, lbls, _ in t_loader:
            imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits, _ = model(imgs, clin)
            loss_fn(logits, lbls).backward(); optimizer.step()
        
        model.eval(); full_dataset.train_mode = False
        wrapped_v_loader = ValLoaderWrapper(v_loader)
        m = val_model(wrapped_v_loader, device, model, loss_fn, v_loader.dataset, threshold=0.5)
        current_f1 = 2*(m[2]*m[3])/(m[2]+m[3]) if (m[2]+m[3])>0 else 0

        if current_f1 > best_f1:
            best_f1, best_metrics_this_split, patience = current_f1, m, 0
            torch.save(model.state_dict(), f"weights/{EXP_NAME}/best_f1_split_{split}.pth")
        else: patience += 1

        print(f"Split {split} Ep {epoch} | F1: {current_f1:.4f} | AUC: {m[5]:.4f} | Acc: {m[1]:.4f}")
        if patience >= 10: break
    
    if best_metrics_this_split is not None: all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY
# ============================================================
final_metrics = np.array(all_split_best_metrics)
avg_metrics, std_metrics = np.mean(final_metrics, axis=0), np.std(final_metrics, axis=0)
print("\n" + "="*45 + "\nFINAL CV SUMMARY (BEST F1 PER SPLIT)\n" + "="*45)
names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
for i, name in enumerate(names):
    print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
print("="*45)

Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping  OFF (pre-dbs updrs)
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Test medication status
Keeping  OFF meds ON stim 6mo
>>> Clinical features normalized successfully.
>>> Loading preprocessed data from cache: qsm_preprocessed_cache.pt...
>>> Pretraining still using 108 total volumes.

SPLIT DATA PROFILE (RAW UNITS)
TRAIN:  52 subs | Responders: 19 ( 36.5%)
VAL:    14 subs | Responders:  8 ( 57.1%)
Mean Age (T/V):      63.2 / 62.7
Mean Duration (T/V): 8.6 / 7.9
Targets (T): {}
Targets (V): {}


>>> Split 0 | PT Subs: 94 | Val Subs: 14

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
PRE-AUGMENTATION COHORT SUMMARY
Total Unique Subjects: 66
Total Non-Responders (0): 5
Total Responders (1):     61
Baseline Response Rate:  92.4%
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

>>> Fine-tuning Classifier Head (Backbone Frozen)
Split 0 Ep 0 | F1: 0.6046 | AUC: 0.6612 | Acc: 0.